# PredNet — Clockwise vs Anticlockwise Comparison
Compares IN and OUT stimuli in both rotation directions.
All four sequences trimmed to equal length before analysis.

Upload before running:
- `fpsi136_410000.pth`
- `MonoWheelIN.mp4`, `MonoWheelOUT.mp4`
- `MonoWheelINAS.mp4`, `MonoWheelOUTAS.mp4`

In [ ]:
!pip install hickle --quiet

In [ ]:
import os, math, glob
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Parameter, functional as F
from torch.nn.modules.utils import _pair
from PIL import Image
import matplotlib.pyplot as plt
import subprocess, cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=1, dilation=1, groups=1, bias=True):
        super().__init__()
        if in_channels % groups != 0: raise ValueError('in_channels must be divisible by groups')
        if out_channels % groups != 0: raise ValueError('out_channels must be divisible by groups')
        kernel_size=_pair(kernel_size); stride=_pair(stride)
        padding=_pair(padding);         dilation=_pair(dilation)
        self.in_channels=in_channels;   self.out_channels=out_channels
        self.kernel_size=kernel_size;   self.stride=stride; self.padding=padding
        self.padding_h=tuple(k//2 for k,s,p,d in zip(kernel_size,stride,padding,dilation))
        self.dilation=dilation;         self.groups=groups
        self.weight_ih=Parameter(torch.Tensor(4*out_channels, in_channels//groups,  *kernel_size))
        self.weight_hh=Parameter(torch.Tensor(4*out_channels, out_channels//groups, *kernel_size))
        self.weight_ch=Parameter(torch.Tensor(3*out_channels, out_channels//groups, *kernel_size))
        if bias:
            self.bias_ih=Parameter(torch.Tensor(4*out_channels))
            self.bias_hh=Parameter(torch.Tensor(4*out_channels))
            self.bias_ch=Parameter(torch.Tensor(3*out_channels))
        else:
            self.register_parameter('bias_ih',None)
            self.register_parameter('bias_hh',None)
            self.register_parameter('bias_ch',None)
        self.register_buffer('wc_blank', torch.zeros(1,1,1,1))
        self.reset_parameters()

    def reset_parameters(self):
        n=4*self.in_channels
        for k in self.kernel_size: n*=k
        stdv=1./math.sqrt(n)
        self.weight_ih.data.uniform_(-stdv,stdv)
        self.weight_hh.data.uniform_(-stdv,stdv)
        self.weight_ch.data.uniform_(-stdv,stdv)
        if self.bias_ih is not None:
            self.bias_ih.data.uniform_(-stdv,stdv)
            self.bias_hh.data.uniform_(-stdv,stdv)
            self.bias_ch.data.uniform_(-stdv,stdv)

    def forward(self, input, hx):
        h_0,c_0=hx
        wx=F.conv2d(input,self.weight_ih,self.bias_ih,self.stride,self.padding,self.dilation,self.groups)
        wh=F.conv2d(h_0,  self.weight_hh,self.bias_hh,self.stride,self.padding_h,self.dilation,self.groups)
        wc=F.conv2d(c_0,  self.weight_ch,self.bias_ch,self.stride,self.padding_h,self.dilation,self.groups)
        wxhc=wx+wh+torch.cat((
            wc[:,:2*self.out_channels],
            self.wc_blank.expand(wc.size(0),wc.size(1)//3,wc.size(2),wc.size(3)),
            wc[:,2*self.out_channels:]),dim=1)
        i=torch.sigmoid(wxhc[:,:self.out_channels])
        f=torch.sigmoid(wxhc[:,self.out_channels:2*self.out_channels])
        g=torch.tanh(   wxhc[:,2*self.out_channels:3*self.out_channels])
        o=torch.sigmoid(wxhc[:,3*self.out_channels:])
        c_1=f*c_0+i*g; h_1=o*torch.tanh(c_1)
        return h_1,(h_1,c_1)

In [ ]:
class SatLU(nn.Module):
    def __init__(self,lower=0,upper=255): super().__init__(); self.lower=lower; self.upper=upper
    def forward(self,x): return F.hardtanh(x,self.lower,self.upper)

class PredNet(nn.Module):
    def __init__(self,R_channels,A_channels,output_mode='error'):
        super().__init__()
        assert len(R_channels)==len(A_channels)
        if output_mode not in ('prediction','error'): raise ValueError('Invalid output_mode')
        self.r_channels=R_channels; self.a_channels=A_channels
        self.n_layers=len(R_channels); self.output_mode=output_mode
        r_above=R_channels[1:]+(0,)
        for l in range(self.n_layers):
            setattr(self,f'cell{l}',ConvLSTMCell(2*A_channels[l]+r_above[l],R_channels[l],kernel_size=3))
        for l in range(self.n_layers):
            conv=nn.Sequential(nn.Conv2d(R_channels[l],A_channels[l],kernel_size=3,padding=1),nn.ReLU())
            if l==0: conv.add_module('satlu',SatLU())
            setattr(self,f'conv{l}',conv)
        self.maxpool=nn.MaxPool2d(kernel_size=2,stride=2)
        self.upsample=nn.Upsample(scale_factor=2)
        for l in range(self.n_layers-1):
            setattr(self,f'update_A{l}',nn.Sequential(
                nn.Conv2d(2*A_channels[l],A_channels[l+1],kernel_size=3,padding=1),self.maxpool))
        self.reset_parameters()

    def reset_parameters(self):
        for l in range(self.n_layers): getattr(self,f'cell{l}').reset_parameters()

    def forward(self,input):
        B,T=input.size(0),input.size(1); H,W=input.size(-2),input.size(-1); dev=input.device
        E_seq,R_seq,H_seq=[],[],[None]*self.n_layers
        for l in range(self.n_layers):
            ds=2**l
            E_seq.append(torch.zeros(B,2*self.a_channels[l],H//ds,W//ds,device=dev))
            R_seq.append(torch.zeros(B,  self.r_channels[l], H//ds,W//ds,device=dev))
        total_error,frame_prediction=[],None
        for t in range(T):
            A=input[:,t].float()
            for l in reversed(range(self.n_layers)):
                cell=getattr(self,f'cell{l}')
                hx=H_seq[l] if H_seq[l] is not None else (R_seq[l],R_seq[l])
                li=E_seq[l] if l==self.n_layers-1 else torch.cat([E_seq[l],self.upsample(R_seq[l+1])],dim=1)
                R_seq[l],H_seq[l]=cell(li,hx)
            for l in range(self.n_layers):
                A_hat=getattr(self,f'conv{l}')(R_seq[l])
                if l==0: frame_prediction=A_hat
                E_seq[l]=torch.cat([F.relu(A_hat-A),F.relu(A-A_hat)],dim=1)
                if l<self.n_layers-1: A=getattr(self,f'update_A{l}')(E_seq[l])
            if self.output_mode=='error':
                total_error.append(torch.cat([e.flatten(1).mean(1,keepdim=True) for e in E_seq],dim=1))
        if self.output_mode=='error': return torch.stack(total_error,dim=2)
        return frame_prediction

In [ ]:
A_channels=(3,48,96,192); R_channels=(3,48,96,192)
model=PredNet(R_channels,A_channels,output_mode='prediction')
ckpt=torch.load('fpsi136_410000.pth',map_location=device)
sd=ckpt['model_state_dict'] if isinstance(ckpt,dict) and 'model_state_dict' in ckpt else ckpt
model.load_state_dict(sd)
model=model.to(device).eval()
print(f'Loaded. Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
CHUNK    = 5
N_SAMPLE = 6

# Keys: (expansion_type, direction)
STIMULI = {
    ('OUT', 'CW'):  'MonoWheelOUT.mp4',
    ('OUT', 'ACW'): 'MonoWheelOUTAS.mp4',
    ('IN',  'CW'):  'MonoWheelIN.mp4',
    ('IN',  'ACW'): 'MonoWheelINAS.mp4',
}

# Colours by expansion type
TYPE_COLOR = {'OUT': '#E24B4A', 'IN': '#378ADD'}
# Line style by direction (for plots)
DIR_STYLE  = {'CW': '-', 'ACW': '--'}

def label(key): return f'{key[0]}-{key[1]}'

def extract_frames(video_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run(['ffmpeg','-y','-i',video_path,'-vf','scale=160:120',
                    '-q:v','2',f'{out_dir}/frame_%04d.jpg'],
                   check=True, capture_output=True)
    return sorted(glob.glob(f'{out_dir}/frame_*.jpg'))

def load_frame(path):
    return np.asarray(Image.open(path)).transpose(2,0,1).astype(np.float32)/255.0

def to_uint8(t):
    return (t.squeeze().permute(1,2,0).clamp(0,1).numpy()*255).astype(np.uint8)

# Extract
raw_frames = {}
for key, vid in STIMULI.items():
    paths = extract_frames(vid, f'frames_{label(key).lower()}')
    raw_frames[key] = [load_frame(p) for p in paths]
    print(f'{label(key)}: {len(paths)} frames')

# Trim all four to the global minimum length
min_len = min(len(f) for f in raw_frames.values())
print(f'\nTrimming all sequences to {min_len} frames')
for key in raw_frames: raw_frames[key] = raw_frames[key][:min_len]

# Build tensors
all_tensors = {}
for key, frames in raw_frames.items():
    t = torch.from_numpy(np.stack(frames)).unsqueeze(0).to(device)
    all_tensors[key] = t
    print(f'{label(key)}: tensor {t.shape}')

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────
results = {}
for key, tensor in all_tensors.items():
    T=tensor.shape[1]; preds,maes=[],[]
    with torch.no_grad():
        for start in range(0, T-1, CHUNK):
            end=min(start+CHUNK+1, T)
            pred=model(tensor[:,start:end])
            preds.append(pred.cpu())
            maes.append(torch.mean(torch.abs(pred.cpu()-tensor[:,end-1].cpu())).item())
    results[key]={'preds':preds,'mae':maes}
    print(f'{label(key)}  chunks={len(preds)}  mean MAE={np.mean(maes):.4f}')

In [ ]:
# ── Wheel mask ────────────────────────────────────────────────────────────
def make_wheel_mask(frame_uint8, threshold=15):
    gray=cv2.cvtColor(frame_uint8,cv2.COLOR_RGB2GRAY)
    _,mask=cv2.threshold(gray,threshold,255,cv2.THRESH_BINARY)
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    return cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel).astype(bool)

first_gt   = to_uint8(all_tensors[('OUT','CW')][:,0].cpu())
wheel_mask = make_wheel_mask(first_gt)
contours,_ = cv2.findContours(wheel_mask.astype(np.uint8),
                               cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
print(f'Mask: {wheel_mask.sum()} inside / {(~wheel_mask).sum()} outside')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────
LK_PARAMS=dict(winSize=(15,15),maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))
FP=dict(maxCorners=80,qualityLevel=0.02,minDistance=5,blockSize=7)

def sample_indices(n_chunks, n_sample):
    return [int(i*(n_chunks-1)/(n_sample-1)) for i in range(n_sample)]

def error_map_rgb(gt, pred, vmax=0.3):
    err=np.mean(np.abs(gt.astype(float)-pred.astype(float)),axis=2)/255.0
    rgb=(plt.cm.hot(err/vmax)[:,:,:3]*255).astype(np.uint8)
    bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
    cv2.drawContours(bgr,contours,-1,(0,200,255),1)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB), err

def draw_lk(img_rgb,p0,p1,status,color):
    out=cv2.cvtColor(img_rgb,cv2.COLOR_RGB2BGR).copy()
    for i,(new,old) in enumerate(zip(p1,p0)):
        if status[i]:
            a,b=new.ravel().astype(int); c,d=old.ravel().astype(int)
            cv2.arrowedLine(out,(c,d),(a,b),color,1,tipLength=0.4)
            cv2.circle(out,(c,d),2,color,-1)
    return cv2.cvtColor(out,cv2.COLOR_BGR2RGB)

def tracked_mag(p0,p1,st):
    v=(p1[st.ravel()==1]-p0[st.ravel()==1]).reshape(-1,2)
    return np.linalg.norm(v,axis=1).mean() if len(v) else 0.0

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Figure 1: GT vs Prediction — OUT (CW top, ACW bottom), then IN same
# Layout: 4 row-pairs x N_SAMPLE cols; pairs grouped by expansion type
# ════════════════════════════════════════════════════════════════════════
KEYS_ORDER = [('OUT','CW'),('OUT','ACW'),('IN','CW'),('IN','ACW')]

fig, axes = plt.subplots(len(KEYS_ORDER)*2, N_SAMPLE,
                         figsize=(3.2*N_SAMPLE, 3.0*len(KEYS_ORDER)))
fig.suptitle('Ground Truth (top of each pair) vs PredNet Prediction (bottom)', fontsize=12, y=1.01)

for s, key in enumerate(KEYS_ORDER):
    tensor=all_tensors[key]; preds=results[key]['preds']; T=tensor.shape[1]
    sidx=sample_indices(len(preds),N_SAMPLE)
    gt_row=s*2; pred_row=s*2+1
    col_color=TYPE_COLOR[key[0]]
    for col, idx in enumerate(sidx):
        fi=min(idx*CHUNK+CHUNK,T-1)
        gt=to_uint8(tensor[:,fi].cpu()); pr=to_uint8(preds[idx])
        axes[gt_row,col].imshow(gt)
        axes[gt_row,col].set_title(f'{label(key)} f{fi}',fontsize=8,color=col_color)
        axes[gt_row,col].axis('off')
        axes[pred_row,col].imshow(pr)
        axes[pred_row,col].set_title(f'MAE {results[key]["mae"][idx]:.3f}',fontsize=8)
        axes[pred_row,col].axis('off')

# Separator lines between OUT and IN groups
for ax in axes[3, :]:
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(2)

plt.tight_layout()
plt.savefig('direction_predictions.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: direction_predictions.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Figure 2: Error maps — same layout
# ════════════════════════════════════════════════════════════════════════
region_stats = {}

fig, axes = plt.subplots(len(KEYS_ORDER), N_SAMPLE,
                         figsize=(3.2*N_SAMPLE, 3.0*len(KEYS_ORDER)))
fig.suptitle('Prediction error map (hot, 0–0.3)  |  cyan = wheel boundary', fontsize=11, y=1.01)

for row, key in enumerate(KEYS_ORDER):
    tensor=all_tensors[key]; preds=results[key]['preds']; T=tensor.shape[1]
    sidx=sample_indices(len(preds),N_SAMPLE)
    ins_list,out_list=[],[]
    for col, idx in enumerate(sidx):
        fi=min(idx*CHUNK+CHUNK,T-1)
        gt=to_uint8(tensor[:,fi].cpu()); pr=to_uint8(preds[idx])
        err_rgb,err=error_map_rgb(gt,pr)
        ins=err[wheel_mask].mean(); out=err[~wheel_mask].mean()
        ins_list.append(ins); out_list.append(out)
        axes[row,col].imshow(err_rgb)
        axes[row,col].set_title(f'{label(key)} f{fi}\nin {ins:.3f} / out {out:.3f}',
                                fontsize=8,color=TYPE_COLOR[key[0]])
        axes[row,col].axis('off')
    region_stats[key]={'inside':np.array(ins_list),'outside':np.array(out_list)}

plt.tight_layout()
plt.savefig('direction_error_maps.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: direction_error_maps.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Figure 3: LK flow — GT and Pred rows per stimulus
# ════════════════════════════════════════════════════════════════════════
lk_stats = {}

fig, axes = plt.subplots(len(KEYS_ORDER)*2, N_SAMPLE,
                         figsize=(3.2*N_SAMPLE, 3.0*len(KEYS_ORDER)))
fig.suptitle('LK sparse flow  —  Top of each pair: GT→GT   Bottom: GT→Pred', fontsize=11, y=1.01)

for s, key in enumerate(KEYS_ORDER):
    tensor=all_tensors[key]; preds=results[key]['preds']; T=tensor.shape[1]
    sidx=sample_indices(len(preds),N_SAMPLE)
    gt_row=s*2; pred_row=s*2+1
    gt_mags,pred_mags=[],[]
    for col, idx in enumerate(sidx):
        fi=min(idx*CHUNK+CHUNK,T-1); pi=max(fi-1,0)
        gp=to_uint8(tensor[:,pi].cpu())
        gc=to_uint8(tensor[:,fi].cpu())
        pc=to_uint8(preds[idx])
        pg=cv2.cvtColor(gp,cv2.COLOR_RGB2GRAY)
        cg=cv2.cvtColor(gc,cv2.COLOR_RGB2GRAY)
        pr=cv2.cvtColor(pc,cv2.COLOR_RGB2GRAY)
        p0=cv2.goodFeaturesToTrack(pg,mask=None,**FP)
        if p0 is None:
            axes[gt_row,col].axis('off'); axes[pred_row,col].axis('off'); continue
        p1g,sg,_=cv2.calcOpticalFlowPyrLK(pg,cg,p0,None,**LK_PARAMS)
        p1p,sp,_=cv2.calcOpticalFlowPyrLK(pg,pr,p0,None,**LK_PARAMS)
        mg=tracked_mag(p0,p1g,sg); mp=tracked_mag(p0,p1p,sp)
        gt_mags.append(mg); pred_mags.append(mp)
        axes[gt_row,col].imshow(draw_lk(gp,p0,p1g,sg,(0,255,100)))
        axes[gt_row,col].set_title(f'{label(key)} GT {mg:.2f}px',
                                   fontsize=8,color=TYPE_COLOR[key[0]])
        axes[gt_row,col].axis('off')
        axes[pred_row,col].imshow(draw_lk(gp,p0,p1p,sp,(255,80,0)))
        axes[pred_row,col].set_title(f'{label(key)} Pred {mp:.2f}px',
                                     fontsize=8,color=TYPE_COLOR[key[0]])
        axes[pred_row,col].axis('off')
    lk_stats[key]={'gt':np.mean(gt_mags),'pred':np.mean(pred_mags)}

plt.tight_layout()
plt.savefig('direction_lk.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: direction_lk.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Figure 4: Summary bar charts — CW vs ACW within each expansion type
# ════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('CW vs ACW comparison within OUT and IN', fontsize=12)

x      = np.arange(2)   # OUT, IN
w      = 0.35
types  = ['OUT', 'IN']
dirs   = ['CW', 'ACW']
hatch  = {'CW': '', 'ACW': '///'}

# Panel 1: inside error
for d in dirs:
    vals=[region_stats[(t,d)]['inside'].mean() for t in types]
    offset = -w/2 if d=='CW' else w/2
    bars=axes[0].bar(x+offset, vals, w, label=d,
                     color=[TYPE_COLOR[t] for t in types],
                     alpha=0.9 if d=='CW' else 0.5, hatch=hatch[d])
axes[0].set_xticks(x); axes[0].set_xticklabels(types)
axes[0].set_ylabel('Mean absolute error (inside wheel)')
axes[0].set_title('Inside-wheel prediction error')
axes[0].legend(dirs); axes[0].grid(axis='y',alpha=0.3)

# Panel 2: inside/outside ratio
for d in dirs:
    ins=[region_stats[(t,d)]['inside'].mean()  for t in types]
    out=[region_stats[(t,d)]['outside'].mean() for t in types]
    ratios=[i/o if o>0 else 0 for i,o in zip(ins,out)]
    offset = -w/2 if d=='CW' else w/2
    axes[1].bar(x+offset, ratios, w, label=d,
                color=[TYPE_COLOR[t] for t in types],
                alpha=0.9 if d=='CW' else 0.5, hatch=hatch[d])
axes[1].axhline(1.0,color='black',linestyle='--',linewidth=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(types)
axes[1].set_ylabel('Inside / outside ratio')
axes[1].set_title('Error ratio: inside vs outside')
axes[1].legend(dirs); axes[1].grid(axis='y',alpha=0.3)

# Panel 3: LK predicted magnitude
for d in dirs:
    vals=[lk_stats[(t,d)]['pred'] for t in types]
    offset = -w/2 if d=='CW' else w/2
    axes[2].bar(x+offset, vals, w, label=d,
                color=[TYPE_COLOR[t] for t in types],
                alpha=0.9 if d=='CW' else 0.5, hatch=hatch[d])
axes[2].set_xticks(x); axes[2].set_xticklabels(types)
axes[2].set_ylabel('Mean LK displacement (px)')
axes[2].set_title('Predicted motion magnitude')
axes[2].legend(dirs); axes[2].grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.savefig('direction_summary_bars.png',dpi=150)
plt.show(); print('Saved: direction_summary_bars.png')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
print('='*65)
print(f'All sequences trimmed to {min_len} frames  |  chunk size = {CHUNK}')
print('='*65)
print(f'{"Stimulus":<12} {"MAE":>8} {"Inside":>8} {"Outside":>9} {"Ratio":>7} {"GT mag":>8} {"Pred mag":>9}')
print('-'*65)
for key in KEYS_ORDER:
    mae =np.mean(results[key]['mae'])
    ins =region_stats[key]['inside'].mean()
    out =region_stats[key]['outside'].mean()
    gmag=lk_stats[key]['gt']; pmag=lk_stats[key]['pred']
    print(f'{label(key):<12} {mae:>8.4f} {ins:>8.4f} {out:>9.4f} {ins/out:>7.2f}x {gmag:>8.3f} {pmag:>9.3f}')